Corrective Rag

In corrective Rag we use three categories appraoch

* MOST RELEVANT: Having atleast one document with relevancy score greater than upper threshold

* LEAST RELEVANT: Having no document relevancy score greater than lower threshold

* AMBIGOUS: Having all the documents relevancy less then upper theshold but atleast one document with relevancy score greater than lower theshold

In first one we do only contexual generation

in the second category we go for only web search

In the ambigous we go for both web and contexual appraoch

In [2]:
from langgraph.graph import StateGraph, START, END
from langchain_openai import ChatOpenAI
from pydantic import BaseModel
from typing import List, TypedDict
from langchain_core.messages import HumanMessage
from langchain_core.documents import Document

from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import TextLoader
import os 
import numpy as np
os.environ["HF_HOME"] = "E:\\huggingface_embedding_cache"
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    encode_kwargs={"normalize_embeddings": True},
)



C:\Users\ndm60\AppData\Local\Temp\ipykernel_16616\1595770610.py:8: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1913.64it/s]


In [ ]:
class State(TypedDict):
    question: str
    documents: list
    filtered_documents: list
    category: str
    web_results: list
    generation: str

In [7]:
from langchain_chroma import Chroma
# ==========================================================
# CREATE VECTOR DATABASE
# ==========================================================
loader = TextLoader(
    "data.txt",
    encoding="utf-8"
)

documents = loader.load()

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks = splitter.split_documents(documents)

persist_directory='./vectorStore'



vectorStore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name="my_docs",
    persist_directory=persist_directory,
)

retriever = vectorStore.as_retriever(
    search_kwargs={"k":3}
)

In [18]:
class Score(BaseModel):
    score : float

llm = ChatOpenAI(
    model="gpt-4.1-mini",  # Your Azure deployment name
    base_url="https://openai-rg-nadeem.openai.azure.com/openai/v1")
structured_llm = llm.with_structured_output(Score)

In [ ]:
def retriever(state: State):
    document = vectorStore.similarity_search(query=state['question'], k=4)
    print(document)
    return{
        "documents": document
    }

def filter(state: State):
    document = state['documents']
    filtered_docs = []
    ambigous_docs= []
    results = []
    category = ""
    for doc in document:
        result = structured_llm.invoke(f'Rate the relevancy of the docuement to the question out of 1 ... from less than 0.3 means very irrelevant and more than .3 and less than .7 means ambiougs and more than 7 menas enough to answer the qeustion understand the question : {state['question']} and the document is {doc}') 
        results.append(result)
        if result >= 0.7:
            filtered_docs.append(doc)
        if result < 0.7 and result >=0.3:
            ambigous_docs.append(doc)
    if filtered_docs:
        category = "relevant"
    elif ambigous_docs:
        category= "ambigous"
        filtered_docs = ambigous_docs
    else:
        filtered_docs = []
        category = "irrelevant"

    return{
        "category": category,
        "filtered_documents": filtered_docs
    }
        
            

[Document(id='358fa25c-12cd-4b20-b0e1-d8037d982dec', metadata={'source': 'data.txt'}, page_content='Google is one of the world\'s largest and most influential technology companies. It was founded on September 4, 1998, by Larry Page and Sergey Brin while they were Ph.D. students at Stanford University. Their mission was simple yet ambitious: **"To organize the world\'s information and make it universally accessible and useful."** Over the years, Google has transformed from a search engine into a multinational technology corporation offering hundreds of products and services across artificial'), Document(id='2922ff31-564f-453b-9fb3-66a90b14bd5a', metadata={'source': 'data.txt'}, page_content='# Google: A Comprehensive Overview'), Document(id='02fa62ec-f9c8-4f75-a008-1e1c7a8835fb', metadata={'source': 'data.txt'}, page_content="## Global Impact\n\nGoogle products are used by billions of people every day. Individuals rely on Google for searching information, navigating cities, communicatin

In [ ]:
from langchain_tavily import TavilySearch
web_search = TavilySearch(max_results=3)
def generate(state: State):
    if state['category']=="relevant":
        results = llm.invoke(f'You have to answer the qustion from the context... the question is {state['question']} and the context is {state['filtered_documents']}... if it don\'t match say sorry I have no answer')
        print(results.content)
        return {
        "generations": results,
        "web_results": ""
    }
    elif state['category'] == "irrelevant":
        web = web_search.invoke(state['question'])
        results = llm.invoke(f'You have to answer the qustion from the context and the web search this is ambigous type of RAG implementation you have to answer that accordingly... the question is {state['question']} and the context is {state['filtered_documents']} and results of web search is {web}... if it don\'t match say sorry I have no answer')
        print(results)
        return{
            "generations": results,
            "web_results": web
        }
    else:
        web = web_search.invoke(state['question'])
        results = llm.invoke(f'You have to answer the qustion from the web search this is irrelevant type of RAG implementation you have to answer that accordingly... the question is {state['question']} and results of web search is {web}... if it don\'t match say sorry I have no answer')
        print(results)
        return{
            "generations": results,
            "web_results": web
        }


In [ ]:
document = vectorStore.similarity_search(query='what is the prime goal of google', k=4)
for doc in document:
    result = structured_llm.invoke(f'Rate the document out of 1 understand the question : {"what are goals of the google"} and the document is {doc}') 
    print(result)
    
    


NameError: name 'state' is not defined